In [1]:
#imports
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import keras

In [2]:
# load data
(ds_in, ds_out) = tfds.load('pneumonia_mnist', split=['train[:50%]', 'train[50%:100%]'], shuffle_files=False, as_supervised=True)

# Change pixel values to be range [0,1]
def normalize_img(image, label):
	"""Normalizes images: `uint8` -> `float32`."""
	return tf.cast(image, tf.float32) / 255., label

#cache and prefetch data for faster performance
ds_in = ds_in.map(
    normalize_img, num_parallel_calls=tf.data.AUTOTUNE)
ds_in = ds_in.cache()
ds_in = ds_in.batch(64)
ds_in = ds_in.prefetch(tf.data.AUTOTUNE)


ds_out = ds_out.map(
    normalize_img, num_parallel_calls=tf.data.AUTOTUNE)
ds_out = ds_out.batch(64)
ds_out = ds_out.cache()
ds_out = ds_out.prefetch(tf.data.AUTOTUNE)

In [3]:
#target model and shadow model architecture
def get_classifier():
	model = tf.keras.models.Sequential([
	tf.keras.layers.Conv2D(128, (3, 3), activation="relu", input_shape=(28, 28, 1)),
	tf.keras.layers.MaxPool2D(2,2),
	tf.keras.layers.Conv2D(128, (3, 3), activation="relu"),
	tf.keras.layers.MaxPool2D(2,2),
	tf.keras.layers.Conv2D(32, (3, 3), activation="relu"),
	tf.keras.layers.MaxPool2D(2,2),
	tf.keras.layers.Flatten(),
	tf.keras.layers.Dense(64, activation='relu'),
	tf.keras.layers.Dense(2)
	])

	return model

In [4]:
#train target model on in data
target_model = get_classifier()

target_model.compile(
optimizer=tf.keras.optimizers.Adam(0.001),
loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
metrics=[tf.keras.metrics.SparseCategoricalAccuracy()],
)

target_model.fit(
    ds_in,
    epochs=25,
	validation_data=ds_out
)

C:\Users\Skylar Marosi\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - loss: 0.5770 - sparse_categorical_accuracy: 0.7285 - val_loss: 0.5401 - val_sparse_categorical_accuracy: 0.7447
Epoch 2/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.4958 - sparse_categorical_accuracy: 0.7472 - val_loss: 0.3855 - val_sparse_categorical_accuracy: 0.7999
Epoch 3/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.3366 - sparse_categorical_accuracy: 0.8488 - val_loss: 0.2983 - val_sparse_categorical_accuracy: 0.8653
Epoch 4/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 0.2874 - sparse_categorical_accuracy: 0.8768 - val_loss: 0.2755 - val_sparse_categorical_accuracy: 0.8874
Epoch 5/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.2486 - sparse_categorical_accuracy: 0.8972 - val_loss: 0.2442 - val_sparse_categorical_accuracy: 0.9010
Epoch 6/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 0.2300 - sparse_categorical_accuracy: 0.9061 - val_loss: 0.2365 - val_sparse_categorical_accuracy: 0.9040
Epoch 7/25

In [5]:
#train and save out-models using mixed loss criteria
mse = keras.losses.MeanSquaredError()
ce = keras.losses.SparseCategoricalCrossentropy()

for i in range(10):
	optimizer = tf.keras.optimizers.Adam(0.001)
	out_model = get_classifier()
	for epoch in range(25):
		ds_out = ds_out.shuffle(buffer_size=10)
		for step, (x_batch_out, y_batch_out) in enumerate(ds_out):
			with tf.GradientTape() as tape:
				student_logits = out_model(x_batch_out, training=False)

				teacher_logits = target_model(x_batch_out)

				regular_loss = ce(y_batch_out, student_logits)

				imitate_loss = mse(teacher_logits, student_logits)

				#half of loss is regular cross entropy, the other half is how far its outputs were from the target model
				loss_value = 0.5 * imitate_loss + 0.5 * regular_loss
				
			grads = tape.gradient(loss_value, out_model.trainable_weights)

			optimizer.apply_gradients(zip(grads, out_model.trainable_weights))
			
	print(f"finished training model {i+1}")
	out_model.save_weights(f'./out_models/out_model_{i+1}.weights.h5')



KeyboardInterrupt: 

In [8]:
#train and save in-models on in data
for i in range(10):
	in_model = get_classifier()

	in_model.compile(
	optimizer=tf.keras.optimizers.Adam(0.001),
	loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
	)

	in_model.fit(
    ds_in,
    epochs=25,
	)

	in_model.save_weights(f'./in_models/in_model_{i+1}.weights.h5')

Epoch 1/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - loss: 1.1025
Epoch 2/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 0.5658
Epoch 3/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 0.5638
Epoch 4/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 0.5399
Epoch 5/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 0.4687
Epoch 6/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.3719
Epoch 7/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.3388
Epoch 8/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 0.2954
Epoch 9/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.2743
Epoch 10/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.2425
Epoch 11/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 0.2397
Epoch 12/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 0.2294
Epoch 13/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 0.2274
Epoch 14/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 0.2189
Epoch 15/25
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 0.2054
Epoc

In [ ]:
#load imitative models for inference
in_models = []
out_models = []
for i in range(10):
	in_models[i] = tf.keras.models.load_model(f'./in_models/in_model_{i+1}.weights.h5')
	out_models[i] = tf.keras.models.load_model(f'./out_models/out_model_{i+1}.weights.h5')

In [ ]:
#calculate scaled confidence score for a batch
def scaled_confidence_score(x, y, model):
	pred = model.predict(x)

	confidences = []
	#calculate scaled confidence score for model(x[i])
	for i in range(len(x)):
		real = max(0.01, pred[i][y[i]])
		other = max(0.01, pred[i][1 - y[i]])
		phi = np.log10(real) - np.log10(other)
		confidences[i] = phi

	return np.array(confidences)

In [ ]:
#make predictions about in data
in_correct = 0
for step, (x_batch_in, y_batch_in) in enumerate(ds_in):
	target_phi = scaled_confidence_score(x_batch_in, y_batch_in, target_model)

	mean_out_phi = np.zeros(len(x_batch_in))
	for i in range(len(out_models)):
		#scaled_confidence_score returns an array where each entry is the model's confidence for that input in the batch, i.e. length 64
		mean_out_phi = mean_out_phi + scaled_confidence_score(x_batch_in, y_batch_in, out_models[i])

	#array of length 64, each entry of which is the mean confidence score between the 10 out models for an entry of x_batch_in
	mean_out_phi = mean_out_phi / len(x_batch_in)

	mean_in_phi = np.zeros(len(x_batch_in))
	for i in range(len(in_models)):
		#scaled_confidence_score returns an array where each entry is the model's confidence for that input in the batch, i.e. length 64
		mean_in_phi = mean_in_phi + scaled_confidence_score(x_batch_in, y_batch_in, in_models[i])

	#array of length 64, each entry of which is the mean confidence score between the 10 in models for an entry of x_batch_in
	mean_in_phi = mean_in_phi / len(x_batch_in)

	pred_lambda = (target_phi - mean_out_phi)**2 - (target_phi - mean_in_phi)**2

	if (pred_lambda > 0):
		in_correct += 1

in_advantage = in_correct / len(ds_in)
print(in_advantage)



#for each query in ds
# find real = phi(f(query))
# find each phi(f_out(query))
# find each phi(f_in(query))
# compute lambda = (real - mean(phi(f_out)))^2 - (real - mean(phi(f_in)))^2
# if lambda > 0 -> predict "in"